# Silver Layer - CBBC Stake
Build the silver CBBC stake dataset for a given trade date by combining the latest records from `bronze.cbbc_summary` and `bronze.cbbc_list`.

In [0]:
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.window import Window

## 1. Set trade date parameter

In [0]:
# Get trade date from widget input
# dbutils.widgets.text("trade_date_str", "")
# trade_date_str = dbutils.widgets.get("trade_date_str")

# Optional local test value
trade_date_str = "2026-01-02"
trade_date = F.to_date(F.lit(trade_date_str), "yyyy-MM-dd")

## 2. Load latest bronze records and prepare silver dataset

In [0]:
# Load latest summary record for each CBBC code
df_summary = (
    spark.table("bronze.cbbc_summary")
         .withColumnRenamed("stock_code", "cbbc_code")
         .withColumn("trade_date", F.to_date("trade_date"))
         .filter(F.col("trade_date") == trade_date)
         .withColumn("cbbc_code", F.regexp_replace("cbbc_code", r"\.0$", ""))
)

w_summary = Window.partitionBy("trade_date", "cbbc_code").orderBy(F.col("processed_at").desc())

df_summary = (
    df_summary
    .withColumn("rn", F.row_number().over(w_summary))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Load latest list record for each CBBC code
df_list = (
    spark.table("bronze.cbbc_list")
         .withColumnRenamed("code", "cbbc_code")
         .withColumn("trade_date", F.to_date("trade_date"))
         .filter(F.col("trade_date") == trade_date)
         .withColumn("cbbc_code", F.regexp_replace("cbbc_code", r"\.0$", ""))
)

w_list = Window.partitionBy("trade_date", "cbbc_code").orderBy(F.col("processed_at").desc())

df_list = (
    df_list
    .withColumn("rn", F.row_number().over(w_list))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Join bronze summary and list datasets
df = df_list.join(df_summary, ["trade_date", "cbbc_code"], "left")

# Clean numeric columns before calculation
numeric_cols = [
    "contracts_sold_count",
    "contracts_bought_count",
    "average_price_per_contract_sold",
    "average_price_per_contract_bought",
    "contracts_outstanding_count",
    "call_level"
]

for c in numeric_cols:
    df = (
        df.withColumn(c, regexp_replace(col(c), ",", ""))
          .withColumn(c, regexp_replace(col(c), r"\((.*)\)", r"-\1"))
          .withColumn(c, col(c).cast("double"))
    )

# Create derived silver metrics
df = (
    df.withColumn(
        "net_contracts_change",
        -col("contracts_sold_count") - col("contracts_bought_count"),
    )
    .withColumn(
        "net_cash_flow",
        col("average_price_per_contract_sold") * col("contracts_sold_count")
        - col("average_price_per_contract_bought") * col("contracts_bought_count"),
    )
)

# Select final silver output columns
df = df.select(
    "trade_date",
    "cbbc_code",
    "bull_bear",
    "call_level",
    "underlying_code",
    "contracts_outstanding_count",
    "listing_date",
    "net_contracts_change",
    "net_cash_flow"
)

## 3. Write output to `silver.cbbc_stake_curated`

In [0]:
# Overwrite the target trade_date partition
(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("replaceWhere", f"trade_date = '{trade_date_str}'")
      .saveAsTable("silver.cbbc_stake_curated")
)

dbutils.notebook.exit(trade_date_str + " Done!")